# 00 · Kaggle setup and correctness checks

Keep this notebook under `notebooks/kaggle/`. Add the project code as a Kaggle input dataset (either the extracted
project or its `polypssl.zip`), enable Internet for the one-time data/weight downloads, and select a GPU.
Code is prepared in `/kaggle/working/polypssl`; data and outputs are isolated under `/kaggle/working/`.
PolypGen and BKAI are optional external evaluation sets; neither is required for the core training experiment.
Do not start training unless all data, weight, and test checks below pass.

In [ ]:
import os
import subprocess
import shutil
import sys
import zipfile
from pathlib import Path, PurePosixPath

PLATFORM = "kaggle"
WORK = Path("/kaggle/working")
CODE_DIR = WORK / "polypssl"
INPUT_DIR = Path("/kaggle/input")
if not INPUT_DIR.is_dir():
    raise FileNotFoundError("This notebook must run in a Kaggle environment")

candidates = sorted({
    path.parent for path in INPUT_DIR.rglob("train.py")
    if (path.parent / "polypssl").is_dir()
})
if len(candidates) == 1:
    shutil.copytree(candidates[0], CODE_DIR, dirs_exist_ok=True)
elif len(candidates) > 1:
    raise RuntimeError(f"Multiple code roots found in Kaggle inputs: {candidates}")
else:
    archives = sorted(INPUT_DIR.rglob("polypssl.zip"))
    if len(archives) != 1:
        raise FileNotFoundError(
            "Add exactly one code input with polypssl.zip or the extracted project folder")
    with zipfile.ZipFile(archives[0]) as zf:
        names = [PurePosixPath(info.filename.replace("\\", "/")) for info in zf.infolist() if not info.is_dir()]
        if any(path.is_absolute() or ".." in path.parts or
               (path.parts and path.parts[0].endswith(":")) for path in names):
            raise ValueError(f"Unsafe path found in {archives[0]}")
        prefix = PurePosixPath("polypssl")
        if not any(path == prefix / "train.py" for path in names):
            prefix = PurePosixPath(".")
        if not any(path == prefix / "train.py" for path in names):
            raise FileNotFoundError("The code ZIP must contain train.py and the polypssl package")
        for info in zf.infolist():
            path = PurePosixPath(info.filename.replace("\\", "/"))
            if info.is_dir() or "__MACOSX" in path.parts:
                continue
            try:
                relative = path.relative_to(prefix) if str(prefix) != "." else path
            except ValueError:
                continue
            if not relative.parts:
                continue
            target = CODE_DIR.joinpath(*relative.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

os.environ["POLYPSSL_ENV"] = PLATFORM
os.environ["POLYPSSL_DATA"] = "/kaggle/working/data"
os.environ["POLYPSSL_OUT"] = "/kaggle/working/outputs"

if not (CODE_DIR / "train.py").is_file():
    raise FileNotFoundError(f"train.py was not prepared in {CODE_DIR}")
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print(f"[{PLATFORM}] code: {CODE_DIR}")
print(f"[{PLATFORM}] data: {os.environ['POLYPSSL_DATA']}")
print(f"[{PLATFORM}] outputs: {os.environ['POLYPSSL_OUT']}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Install/check datasets and checkpoints

The setup inspects each archive and validates matching image/mask stems before extraction. It fails on missing, extra, or duplicate files. PolypGen and BKAI are optional; their external layouts are not part of the core preflight.

In [ ]:
subprocess.run([sys.executable, "setup_env.py"], check=True)

## Unit tests

In [ ]:
from polypssl.paths import get_locations
os.environ["POLYPSSL_TEST_WEIGHTS"] = str(get_locations().weights)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)

## Label splits

In [ ]:
!ls splits

## Optional GPU speed checks

In [ ]:
!python speed_test.py --config configs/stageA/fulnet_c_mscan_full.yaml --iterations 50
!python speed_test.py --config configs/stageA/polyp_pvt_reproduce.yaml --iterations 50
!python speed_test.py --config configs/stageA/fulnet_c_pvt_full.yaml --iterations 50